In [38]:
import os
import re
from pathlib import Path

import geopandas as gpd
import ipywidgets as widgets
import lasio as ls
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import LogNorm, to_rgba
from matplotlib.patches import Patch
from scipy.spatial import cKDTree


In [31]:
# ---------------------------------------------------------------------
# 0) Paths and input files
# ---------------------------------------------------------------------
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

csv_path = Path("./outputs/merged_inversion_results/merged_iteration_14_resistivity_wide.csv")
if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found: {csv_path}")

# well_data_dir = Path("data/well_data")
well_data_dir = Path("data/aer/well")
# water_well_shp = well_data_dir / "WaterWell_HQ_LAS/WaterWell_HQ_LAS/Shapefile/WWLAS_CLC_HQ.shp"
# coal_hole_shp = well_data_dir / "CoalHoles_HQ_LAS/CoalHoles_HQ_LAS/ShapeFile/CoalHole_CLC_HQ.shp"
water_well_shp = well_data_dir / "WaterWell_HQ_LAS/Shapefile/WWLAS_CLC_HQ.shp"
coal_hole_shp = well_data_dir / "CoalHoles_HQ_LAS/ShapeFile/CoalHole_CLC_HQ.shp"
print(water_well_shp)
if not water_well_shp.exists() or not coal_hole_shp.exists():
    raise FileNotFoundError("Well shapefile(s) not found under data/well_data")


data/aer/well/WaterWell_HQ_LAS/Shapefile/WWLAS_CLC_HQ.shp


In [ ]:
test:list = ["rho_(00)","rho_33:."]

In [ ]:
# ---------------------------------------------------------------------
# 1) Load iteration-14 inversion CSV
# ---------------------------------------------------------------------
iter14_df = pd.read_csv(csv_path)

# Numeric cleanup
for col in ["x_wgs84", "y_wgs84", "distance", "dtm", "bheight"]:
    if col in iter14_df.columns:
        iter14_df[col] = pd.to_numeric(iter14_df[col], errors="coerce") # "coerce" set invalid values to NaN.
if "Line" in iter14_df.columns:
    iter14_df["Line"] = iter14_df["Line"].astype(str)

rho_cols = [c for c in iter14_df.columns if c.startswith("rho_") and c.endswith("m")]
if len(rho_cols) == 0:
    raise ValueError("No resistivity columns found in CSV (expected rho_XX.XXm columns)")

# Get depth values from column names with "regex".
def parse_depth_from_col(col_name):
    # (...) is a capturing group (create a groups).
    # [0-9]+ matches any digit from 0 to 9, one or more.
    # (?:...) is a non-capturing group (don't create a group).
    # \.[0-9]+ matches with . followed by one or more digits.
    match = re.match(r"rho_([0-9]+(?:\.[0-9]+)?)m", col_name)
    return float(match.group(1)) if match else np.nan

depth_values = np.array([parse_depth_from_col(c) for c in rho_cols], dtype=float)
sort_idx = np.argsort(depth_values)
rho_cols = [rho_cols[i] for i in sort_idx]
depth_values = depth_values[sort_idx]

# Capture valid x-/y- coordinates.
xy = iter14_df[["x_wgs84", "y_wgs84"]].to_numpy(dtype=float)
xy_valid_mask = np.isfinite(xy).all(axis=1)
if not np.any(xy_valid_mask):
    raise ValueError("No valid x_wgs84 / y_wgs84 values found in CSV")

# Create a KDTree for fast nearest-neighbor search of valid x-/y- coordinates.
iter14_valid = iter14_df.loc[xy_valid_mask].reset_index(drop=True)
xy_valid = iter14_valid[["x_wgs84", "y_wgs84"]].to_numpy(dtype=float)
model_tree = cKDTree(xy_valid)

# Robust color range for map
all_rho = iter14_valid[rho_cols].to_numpy(dtype=float)
valid_rho = all_rho[np.isfinite(all_rho) & (all_rho > 0)]

In [59]:
gpd.read_file("./data/aer/well/WaterWell_HQ_LAS/Tables/WWLAS_CLC_HQ_collar.dbf").head(1)

,LAS_FI_NAM,GIC_W_ID,LONG_NAD83,LAT_NAD83,UTM83_Z,E_UTM83,N_UTM83,E_10TM83,N_10TM83,EL_DEM15_M,DLS_LOC,CURVE_TYPE,SHORTNAME,PUBLISHER,SHORTNAME_
0,153840-a,153840.0,113.69356,50.17612,12.0,307679.0,5561685.0,593284.0,5559029.0,1062.0,151301428M4,DEPTH BELOW REFERENCE; RESISTIVITY; GAMMA RAY,DEPT; RES; GR,Alberta Geological Survey,3.0


In [58]:
gpd.read_file("./data/aer/well/WaterWell_HQ_LAS/Shapefile/WWLAS_CLC_HQ.shp").head(1)

,LAS_FI_NAM,GIC_W_ID,LONG_NAD83,LAT_NAD83,UTM83_Z,E_UTM83,N_UTM83,E_10TM83,N_10TM83,EL_DEM15_M,DLS_LOC,CURVE_TYPE,SHORTNAME,PUBLISHER,SHORTNAME_,geometry
0,153840-a,153840.0,113.69356,50.17612,12.0,307679.0,5561685.0,593284.0,5559029.0,1062.0,151301428M4,DEPTH BELOW REFERENCE; RESISTIVITY; GAMMA RAY,DEPT; RES; GR,Alberta Geological Survey,3.0,POINT Z (593284 5559029 1062)


In [53]:
a = ls.read(well_data_dir / "WaterWell_HQ_LAS/LAS_Files/153847-a.las")
a.df().head()

,RES,GR
DEPT,,
0.25,NaN,80.5073
0.50,NaN,80.5280
0.75,NaN,80.5798
1.00,NaN,80.6315
1.25,NaN,80.6936


In [ ]:
# ---------------------------------------------------------------------
# 2) Load wells and build a single well table (UTM12 or NAD83)
# ---------------------------------------------------------------------
ww_gdf = gpd.read_file(water_well_shp).to_crs(epsg=26912)
ch_gdf = gpd.read_file(coal_hole_shp).to_crs(epsg=26912)


def pick_name(row, candidates):
    for col in candidates:
        if col in row.index and not pd.isna(row[col]):
            text = str(row[col]).strip()
            if text != "":
                return text
    return None

# Extract meta data of well. (well type, file name, x, y)
well_records = []
for _, row in ww_gdf.iterrows():
    if row.geometry is None or row.geometry.is_empty:
        continue
    name = pick_name(
        row, 
        [
            "LAS_FI_NAM",
            "Name",         
            "SHORTNAME"])
    if name is None:
        continue
    well_records.append(
        {
            "well_type": "WaterWell",
            "well_name": name,
            "x": float(row.geometry.x),
            "y": float(row.geometry.y),
        }
    )

# Extract meta data of coal hole. (well type, file name, x, y)
for _, row in ch_gdf.iterrows():
    if row.geometry is None or row.geometry.is_empty:
        continue
    name = pick_name(row, ["Name", "LAS_FI_NAM", "Name_1"])
    if name is None:
        continue
    well_records.append(
        {
            "well_type": "CoalHole",
            "well_name": name,
            "x": float(row.geometry.x),
            "y": float(row.geometry.y),
        }
    )

well_df = pd.DataFrame(well_records)
if well_df.empty:
    raise ValueError("No valid wells were found in shapefiles")

well_df["well_id"] = well_df["well_type"] + " | " + well_df["well_name"]

# Nearest inversion model for each well
well_xy = well_df[["x", "y"]].to_numpy(dtype=float)
nearest_dist, nearest_model_idx = model_tree.query(well_xy)
well_df["nearest_model_idx"] = nearest_model_idx
well_df["nearest_model_dist_m"] = nearest_dist
well_df["well_name_key"] = well_df["well_name"].astype(str).str.strip().str.upper()


239476-a


In [45]:
# ---------------------------------------------------------------------
# 3) Optional well log and lithology readers (overlay in right panel)
# ---------------------------------------------------------------------
water_lith_path = well_data_dir / "WaterWell_HQ_LAS/Tables/WWLAS_CLC_HQ_lithology.DBF"
water_collar_path = well_data_dir / "WaterWell_HQ_LAS/Tables/WWLAS_CLC_HQ_collar.dbf"
coal_lith_path = well_data_dir / "CoalHoles_HQ_LAS/Tables/CoalHole_CLC_HQ_Lithology.txt"

water_collar_df = gpd.read_file(water_collar_path)[["LAS_FI_NAM", "GIC_W_ID"]].copy()
water_collar_df["GIC_W_ID"] = pd.to_numeric(water_collar_df["GIC_W_ID"], errors="coerce")
water_collar_df["well_name"] = water_collar_df["LAS_FI_NAM"].astype(str).str.strip()


def classify_broad_lithology(*parts):
    text = " ".join(str(part).strip().lower() for part in parts if pd.notna(part) and str(part).strip())
    if text == "":
        return "Unknown"
    if "no recovery" in text or "no return" in text:
        return "No recovery"
    if "bentonite" in text or "bent" in text:
        return "Bentonite"
    if "carbonaceous" in text or "carb" in text:
        return "Carbonaceous"
    if "coal" in text:
        return "Coal"
    if "till" in text or "overburden" in text or "topsoil" in text:
        return "Till/Overburden"
    if "clay" in text:
        return "Clay"
    if "sandstone" in text or "sand" in text or "ss" in text:
        return "Sand/Sandstone"
    if "siltstone" in text or "silty" in text or "silt" in text or "slt" in text:
        return "Silt/Siltstone"
    if "mudstone" in text or "shale" in text or "sh" in text:
        return "Shale/Mudstone"
    return "Other"

water_lith_df = gpd.read_file(water_lith_path).copy()
water_lith_df["GIC_WELL_I"] = pd.to_numeric(water_lith_df["GIC_WELL_I"], errors="coerce")
water_lith_df = water_lith_df.merge(
    water_collar_df[["GIC_W_ID", "well_name"]],
    left_on="GIC_WELL_I",
    right_on="GIC_W_ID",
    how="left",
)
water_lith_df["top_m"] = pd.to_numeric(water_lith_df["FINALFROM_"], errors="coerce")
water_lith_df["bottom_m"] = pd.to_numeric(water_lith_df["FINALTO_M"], errors="coerce")
water_lith_df["raw_lithology"] = water_lith_df["MATERIAL"].fillna("Unknown").astype(str).str.strip()
water_lith_df["display_label"] = water_lith_df["raw_lithology"].apply(classify_broad_lithology)
water_lith_df["lith_code"] = water_lith_df["display_label"].str.lower()
water_lith_df["color_hint"] = water_lith_df["COLOUR"]
water_lith_df["well_type"] = "WaterWell"

coal_lith_df = pd.read_csv(
    coal_lith_path,
    header=None,
    names=[
        "well_name",
        "record_id",
        "from_ft",
        "to_ft",
        "depth_unit",
        "top_m",
        "bottom_m",
        "top_m_dup",
        "bottom_m_dup",
        "lith_primary",
        "lith_secondary",
        "rock_color",
        "description",
        "extra",
    ],
)
coal_lith_df["well_name"] = coal_lith_df["well_name"].astype(str).str.strip()
coal_lith_df["top_m"] = pd.to_numeric(coal_lith_df["top_m"], errors="coerce")
coal_lith_df["bottom_m"] = pd.to_numeric(coal_lith_df["bottom_m"], errors="coerce")
coal_lith_df["raw_lithology"] = coal_lith_df.apply(
    lambda row: " / ".join(
        [
            value
            for value in [str(row["lith_primary"]).strip(), str(row["lith_secondary"]).strip(), str(row["description"]).strip()]
            if value and value.lower() != "nan"
        ]
    ) or "Unknown",
    axis=1,
)
coal_lith_df["display_label"] = coal_lith_df["raw_lithology"].apply(classify_broad_lithology)
coal_lith_df["lith_code"] = coal_lith_df["display_label"].str.lower()
coal_lith_df["color_hint"] = np.nan
coal_lith_df["well_type"] = "CoalHole"

lithology_df = pd.concat(
    [
        water_lith_df[["well_type", "well_name", "top_m", "bottom_m", "lith_code", "display_label", "color_hint"]],
        coal_lith_df[["well_type", "well_name", "top_m", "bottom_m", "lith_code", "display_label", "color_hint"]],
    ],
    ignore_index=True,
)
lithology_df = lithology_df.loc[
    lithology_df["well_name"].notna()
    & np.isfinite(lithology_df["top_m"])
    & np.isfinite(lithology_df["bottom_m"])
    & (lithology_df["bottom_m"] > lithology_df["top_m"])
].copy()
lithology_df["well_name_key"] = lithology_df["well_name"].astype(str).str.strip().str.upper()
lithology_df["lookup_key"] = lithology_df["well_type"] + "|" + lithology_df["well_name_key"]

lith_palette = plt.get_cmap("tab20").colors
lithology_color_map = {
    label: lith_palette[index % len(lith_palette)]
    for index, label in enumerate(sorted(lithology_df["lith_code"].dropna().unique()))
}
lithology_label_map = {lith_code: str(lith_code).replace('_', ' ').title() for lith_code in sorted(lithology_color_map)}
lithology_legend_handles = [
    Patch(
        facecolor=to_rgba(lithology_color_map[lith_code], alpha=0.45),
        edgecolor="white",
        linewidth=0.4,
        label=lithology_label_map[lith_code],
    )
    for lith_code in sorted(lithology_color_map)
]
lithology_by_well = {
    key: group.sort_values(["top_m", "bottom_m"]).reset_index(drop=True)
    for key, group in lithology_df.groupby("lookup_key")
}


def make_well_lookup_key(well_name, well_type):
    return f"{well_type}|{str(well_name).strip().upper()}"


def load_well_lithology(well_name, well_type):
    return lithology_by_well.get(make_well_lookup_key(well_name, well_type))


def lithology_facecolor(lith_code, color_hint=None):
    return to_rgba(lithology_color_map.get(lith_code, (0.7, 0.7, 0.7)), alpha=0.45)


def load_well_las(well_name, well_type):
    if well_type == "WaterWell":
        las_path = Path(
            well_data_dir / f"WaterWell_HQ_LAS/LAS_Files/{well_name}.las"
        )
    else:
        las_path = Path(
            well_data_dir / f"CoalHoles_HQ_LAS/LAS_Files/Litholog_{well_name}.las"
        )
        if not las_path.exists():
            las_path = Path(
                well_data_dir / f"CoalHoles_HQ_LAS/LAS_Files/Litholog_{well_name}_A.las"
            )

    if not las_path.exists():
        return None, None

    las = ls.read(las_path)
    df_las = las.df()
    depth = np.asarray(df_las.index.values, dtype=float)

    units = las.index_unit
    if units is None and len(las.curves) > 0:
        units = las.curves[0].unit
    if units is not None:
        units = str(units).upper()
        if "FT" in units or "FEET" in units:
            depth = depth * 0.3048

    res = None
    # CILD: Conductivity Induction Log Deep ()
    for col in ["RES", "RES1", "RES2", "CILD"]:
        if col in df_las.columns:
            values = np.asarray(df_las[col].values, dtype=float)
            if col == "CILD":
                values = 1000.0 / (values + 1e-6)
            res = values
            break

    if res is None:
        return None, None

    mask = np.isfinite(depth) & np.isfinite(res) & (depth >= 0) & (res > 0) & (res < 1e5)
    if not np.any(mask):
        return None, None
    return depth[mask], res[mask]



In [36]:
vmin = 8
vmax = 60

In [ ]:
# ---------------------------------------------------------------------
# 4) Interactive visualization (pcolormesh depth slice)
# ---------------------------------------------------------------------
# Precompute map grid and IDW interpolation weights once
map_dx = 100.0
map_dy = 100.0
x_pad = 1000.0
y_pad = 1000.0
k_nearest_points = 100
max_distance = 500.0
idw_power = 1.0

xmin = np.nanmin(xy_valid[:, 0]) - x_pad
xmax = np.nanmax(xy_valid[:, 0]) + x_pad
ymin = np.nanmin(xy_valid[:, 1]) - y_pad
ymax = np.nanmax(xy_valid[:, 1]) + y_pad

nx = int(np.floor((xmax - xmin) / map_dx)) + 1
ny = int(np.floor((ymax - ymin) / map_dy)) + 1
x_map = np.arange(nx) * map_dx + xmin
y_map = np.arange(ny) * map_dy + ymin
X_map, Y_map = np.meshgrid(x_map, y_map)

map_tree = cKDTree(xy_valid)
k_idw = min(int(k_nearest_points), len(xy_valid))
dist_idw, idx_idw = map_tree.query(np.c_[X_map.ravel(), Y_map.ravel()], k=k_idw)
if k_idw == 1:
    dist_idw = dist_idw[:, None]
    idx_idw = idx_idw[:, None]

idw_eps = min(map_dx, map_dy)
idw_weights = 1.0 / ((dist_idw + idw_eps) ** idw_power)
nearest_distance, _ = map_tree.query(np.c_[X_map.ravel(), Y_map.ravel()], k=1)
mask_outside_data = nearest_distance > max_distance


def interpolate_depth_to_map(values_1d):
    values_1d = np.asarray(values_1d, dtype=float)
    grid_flat = np.sum(idw_weights * values_1d[idx_idw], axis=1) / np.sum(idw_weights, axis=1)
    grid_flat[mask_outside_data] = np.nan
    return grid_flat.reshape(X_map.shape)


depth_slider = widgets.SelectionSlider(
    options=[(f"{d:.2f} m", int(i)) for i, d in enumerate(depth_values)],
    value=0,
    description="Depth:",
    continuous_update=False,
)

well_ids = sorted(well_df["well_id"].tolist())
well_dropdown = widgets.Dropdown(
    options=[(w, w) for w in well_ids],
    value=well_ids[0],
    description="Well:",
)


def plot_depth_slice_with_well(depth_index, well_id):
    depth = float(depth_values[depth_index])
    rho_col = rho_cols[depth_index]

    selected = well_df[well_df["well_id"] == well_id].iloc[0]
    well_lith = load_well_lithology(selected["well_name"], selected["well_type"])
    near_idx = int(selected["nearest_model_idx"])
    near_dist = float(selected["nearest_model_dist_m"])
    model_row = iter14_valid.iloc[near_idx]
    model_profile = model_row[rho_cols].to_numpy(dtype=float)

    fig, ax = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [3, 1.5]})

    # Left panel: pcolormesh depth slice + wells + nearest model
    rho_depth = iter14_valid[rho_col].to_numpy(dtype=float)
    rho_grid = interpolate_depth_to_map(rho_depth)

    pcm = ax[0].pcolormesh(
        x_map,
        y_map,
        rho_grid,
        cmap="turbo",
        norm=LogNorm(vmin=vmin, vmax=vmax),
        shading="auto",
    )
    cbar = plt.colorbar(pcm, ax=ax[0], pad=0.01)
    cbar.set_label("Resistivity (ohm-m)")

    # Optional raw sounding points for spatial context
    # ax[0].plot(xy_valid[:, 0], xy_valid[:, 1], "k,", alpha=0.08)

    water_mask = well_df["well_type"] == "WaterWell"
    coal_mask = well_df["well_type"] == "CoalHole"
    ax[0].scatter(
        well_df.loc[water_mask, "x"],
        well_df.loc[water_mask, "y"],
        c="magenta",
        s=70,
        alpha=0.6,
        label="Water wells",
    )
    ax[0].scatter(
        well_df.loc[coal_mask, "x"],
        well_df.loc[coal_mask, "y"],
        c="black",
        s=70,
        alpha=0.6,
        label="Coal holes",
    )

    ax[0].scatter(
        [selected["x"]],
        [selected["y"]],
        marker="*",
        c="gold",
        s=260,
        edgecolors="black",
        linewidths=0.9,
        zorder=6,
        label="Selected well",
    )

    ax[0].scatter(
        [model_row["x_wgs84"]],
        [model_row["y_wgs84"]],
        marker="x",
        c="red",
        s=70,
        linewidths=2.0,
        zorder=6,
        label="Nearest model",
    )

    ax[0].set_aspect(1)
    ax[0].set_xlabel("x_wgs84")
    ax[0].set_ylabel("y_wgs84")
    ax[0].set_title(f"Iteration 14 Depth Slice: {depth:.2f} m")
    ax[0].legend(loc="best", fontsize=9)
    ax[0].grid(False)

    # Right panel: nearest inversion profile + current depth marker
    valid_profile = np.isfinite(model_profile) & (model_profile > 0)
    ax[1].step(
        model_profile[valid_profile],
        depth_values[valid_profile],
        "-",
        color="blue",
        lw=2.0,
        label="Nearest inverted model",
    )

    # Overlay well log if available
    well_depth, well_res = load_well_las(selected["well_name"], selected["well_type"])
    if well_depth is not None and well_res is not None:
        well_mask = np.isfinite(well_depth) & np.isfinite(well_res) & (well_res > 0)
        if np.any(well_mask):
            ax[1].plot(
                well_res[well_mask],
                well_depth[well_mask],
                "k-",
                lw=1.1,
                alpha=0.75,
                label="Well log",
            )

    if well_lith is not None and not well_lith.empty:
        for interval in well_lith.itertuples(index=False):
            interval_color = lithology_facecolor(interval.lith_code, interval.color_hint)
            ax[1].axhspan(
                interval.top_m,
                interval.bottom_m,
                xmin=0.0,
                xmax=0.12,
                facecolor=interval_color,
                edgecolor="white",
                linewidth=0.4,
                zorder=0,
            )
            if (interval.bottom_m - interval.top_m) >= 6.0:
                ax[1].text(
                    0.14,
                    0.5 * (interval.top_m + interval.bottom_m),
                    interval.display_label,
                    transform=ax[1].get_yaxis_transform(),
                    va="center",
                    ha="left",
                    fontsize=7,
                    color="black",
                    clip_on=True,
                )
        ax[1].text(0.0, 1.02, "Lithology", transform=ax[1].transAxes, fontsize=8)

    rho_at_depth = float(model_profile[depth_index])
    if np.isfinite(rho_at_depth) and rho_at_depth > 0:
        ax[1].scatter(
            [rho_at_depth],
            [depth],
            marker="*",
            c="gold",
            s=220,
            edgecolors="black",
            linewidths=0.9,
            zorder=7,
            label=f"Selected depth {depth:.2f} m",
        )

    ax[1].set_xscale("log")
    ax[1].invert_yaxis()
    ax[1].set_xlabel("Resistivity (ohm-m)")
    ax[1].set_ylabel("Depth (m)")
    ax[1].set_title(
        f"Nearest Model Profile\n{selected['well_id']}\nDistance: {near_dist:.1f} m"
    )
    ax[1].grid(True, which="both", ls="--", alpha=0.35)
    ax[1].legend(loc="upper right", fontsize=8)
    ax[1].set_xlim(1,1e3)
    plot_depth_limits = [depth_values.max(), 350.0]
    if well_depth is not None and len(well_depth) > 0:
        plot_depth_limits.append(np.nanmax(well_depth))
    if well_lith is not None and not well_lith.empty:
        plot_depth_limits.append(float(well_lith["bottom_m"].max()))
    ax[1].set_ylim(max(plot_depth_limits) * 1.03, -20)

    lith_ncol = 1 if len(lithology_legend_handles) <= 8 else 2 if len(lithology_legend_handles) <= 16 else 3
    fig.legend(
        handles=lithology_legend_handles,
        title="Lithology colors",
        loc="lower center",
        bbox_to_anchor=(0.8, -0.03),
        fontsize=7,
        title_fontsize=8,
        ncol=lith_ncol,
        framealpha=0.95,
    )
    plt.tight_layout(rect=[0, 0.1, 1, 1])
    plt.show()


viz_out = widgets.interactive_output(
    plot_depth_slice_with_well,
    {
        "depth_index": depth_slider,
        "well_id": well_dropdown,
    },
)

display(widgets.VBox([depth_slider, well_dropdown, viz_out]))

print(f"Loaded iteration CSV: {csv_path}")
print(f"Soundings: {len(iter14_valid):,}, Depth channels: {len(rho_cols)}")
print(f"Wells: {len(well_df):,}")
print(f"Map grid: {X_map.shape}, IDW neighbors: {k_idw}")

Loaded iteration CSV: outputs/merged_inversion_results/merged_iteration_14_resistivity_wide.csv
Soundings: 561,475, Depth channels: 22
Wells: 77
Map grid: (2053, 1511), IDW neighbors: 100
